# bindcurve basics

This notebook shows the core workflow: build `DoseResponseData`, fit a model, inspect `FitResults`, and plot curves/residuals. The data are synthetic.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import bindcurve as bc

rng = np.random.default_rng(123)

## Create long-form dose-response data

The canonical columns are `compound_id`, `experiment_id`, `concentration`, `replicate_id`, and `response`.

In [ ]:
def inhibition_curve(x, ymin=0.0, ymax=100.0, IC50=1.5, hill_slope=-1.1):
    return ymin + (ymax - ymin) / (1.0 + (IC50 / x) ** hill_slope)

rows = []
concentrations = np.logspace(-2, 2, 12)
for experiment_id, ic50_scale in {'exp1': 0.9, 'exp2': 1.0, 'exp3': 1.1}.items():
    for concentration in concentrations:
        mean = inhibition_curve(concentration, IC50=1.5 * ic50_scale)
        for replicate_id in range(1, 4):
            rows.append({
                'compound_id': 'compound_a',
                'experiment_id': experiment_id,
                'concentration': concentration,
                'replicate_id': f'rep{replicate_id}',
                'response': mean + rng.normal(0.0, 2.0),
            })

raw = pd.DataFrame(rows)
raw.head()

## Build `DoseResponseData` and fit

By default, bindcurve fits one curve per independent experiment and pools technical replicates at each concentration.

In [ ]:
data = bc.DoseResponseData.from_dataframe(raw, concentration_unit='uM', response_unit='percent')

results = bc.fit(
    data,
    model='ic50',
    fixed={'ymin': 0.0, 'ymax': 100.0},
)

results.fits_to_dataframe()

In [ ]:
results.summary_to_dataframe()

## Plot curves and residuals

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
bc.plot_curves(
    data,
    results,
    ax=ax,
    confidence_band=True,
    show_asymptotes=True,
    curve_points=[(1.5, 'nominal IC50')],
)
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
bc.plot_residuals(data, results, ax=ax)
ax.legend()
plt.show()